In [ ]:
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

sys.path.append("../src")
from diagnostic_toolkit import (
    full_metric_analysis,
    mae_metric,
    mase_metric,
    mbe_metric,
    mse_metric,
    plot_results,
)
from feature_engineering import holiday_features, prepare_features, weekly_yearly_features
from forecasting import create_features_and_targets, create_time_series_splits
from xgboost_model import NHSForecaster

In [ ]:
df = pd.read_csv("../src/nhs_ead_forecast_training_validation_dataset.csv")

In [ ]:
forecasting_labels = list(df.columns.values)

target_label = ["estimated_avoidable_deaths"]

if forecasting_labels is None:
    forecasting_df = df[["midday_day"] + target_label]
    forecasting_exog = None
else:
    forecasting_df = df  # df[["midday_day"] + forecasting_labels + target_label]
    forecasting_exog = df[forecasting_labels]

forecasting_df["midday_day"] = pd.to_datetime(forecasting_df["midday_day"]).dt.date

In [ ]:
forecasting_df = forecasting_df.bfill()
forecasting_df = weekly_yearly_features(forecasting_df)
forecasting_df = holiday_features(forecasting_df)

In [ ]:
X, y = create_features_and_targets(
    forecasting_df, feature_lags=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 14, 21), target_lags=(3, 4, 5, 6, 7, 10, 14, 21)
)

In [ ]:
n_splits = 20
splits = create_time_series_splits(X, use_cross_validation=True, n_splits=n_splits, gap=0)

In [ ]:
train_indices = [split[0] for split in splits]
test_indices = [split[1] for split in splits]
X_train = [prepare_features(X, idx, X.columns) for idx in train_indices]
# X_train = [X.iloc[idx] for idx in train_indices]
y_train = [y.iloc[idx] for idx in train_indices]
X_test = [prepare_features(X, idx, X.columns) for idx in test_indices]
# X_test = [X.iloc[idx] for idx in test_indices]
y_test = [y.iloc[idx] for idx in test_indices]

In [ ]:
split = 10

X_train_fix, y_train_fix, X_test_fix, y_test_fix = X_train[split], y_train[split], X_test[split], y_test[split]

In [ ]:
nhs_model = NHSForecaster()

In [ ]:
X_train_fix

In [ ]:
trained_models = nhs_model.train_models(X_train_fix, y_train_fix)

In [ ]:
todays_values = X_test_fix.iloc[[0]]

forecast_df = nhs_model.forecast(todays_values)

In [ ]:
pred_vals = forecast_df.values
true_vals = y_test_fix.iloc[0].values

In [ ]:
mean_squared_error(pred_vals, true_vals)

In [ ]:
plt.plot(pred_vals, label="pred")
plt.plot(true_vals, label="true")
plt.legend()

## All splits of the data ##

In [ ]:
y_train_target = [df["estimated_avoidable_deaths"][idx] for idx in train_indices]
y_test_target = [df["estimated_avoidable_deaths"][idx] for idx in test_indices]
y_pred_target = []
y_pred_models = []

In [ ]:
for split in range(0, n_splits):
    X_train_fix, y_train_fix, X_test_fix, y_test_fix = X_train[split], y_train[split], X_test[split], y_test[split]
    trained_models = nhs_model.train_models(X_train_fix, y_train_fix)
    y_pred_models.append(trained_models)
    todays_values = X_test_fix.iloc[[0]]
    forecast_df = nhs_model.forecast(todays_values)
    y_pred_target.append(forecast_df.values)

In [ ]:
# Save XGBoost models
joblib.dump(y_pred_models, "xgboost_20Splits_maxdepth5_models_improve.pkl")

In [ ]:
(mse_list, summed_mse, mse_1_5_list, summed_mse_1_5, mse_6_10_list, summed_mse_6_10) = mse_metric(
    y_test_target, y_pred_target
)
(mae_list, summed_mae, mae_1_5_list, summed_mae_1_5, mae_6_10_list, summed_mae_6_10) = mae_metric(
    y_test_target, y_pred_target
)
mbe_list, mbe_1_5_list, mbe_6_10_list = mbe_metric(y_test_target, y_pred_target)
mase_list, mase_1_5_list, mase_6_10_list = mase_metric(y_train_target, y_test_target, y_pred_target, 7)

In [ ]:
# Extract the last value from each array as your index list
last_train_indices = [arr[-1] for arr in train_indices]

# Map to the NHS datetime reference
datetime_refs = [df["midday_day"].iloc[i] for i in last_train_indices]

results_list, results_single = full_metric_analysis(y_train_target, y_test_target, y_pred_target, 7)

results_list.index = pd.to_datetime(datetime_refs).date

In [ ]:
## Load data if already created ##
# results_list = pd.read_csv('xgboost_50Splits_maxdepth5.csv')

In [ ]:
results_list

In [ ]:
plot_results(results_list)

In [ ]:
results_list.describe()

In [ ]:
results_list[["mse", "mae", "mbe", "mase"]].boxplot()

In [ ]:
results_list.to_csv("xgboost_20Splits_maxdepth5_improve.csv")

In [ ]:
def mae_metric_day(test_data: list[pd.DataFrame], forecast_data: list[pd.DataFrame]) -> list[list[float], list[float]]:

    if len(test_data) != len(forecast_data):
        raise ValueError("testing and forecast data sets must be same length.")

    mae_list = []
    for k in range(len(test_data)):
        mae_list.append(np.abs(test_data[k] - forecast_data[k]))

    return mae_list

In [ ]:
mae_list = mae_metric_day(y_test_target, y_pred_target)

In [ ]:
mae_list = np.array(mae_list)

In [ ]:
plt.boxplot(mae_list)
plt.xlabel("Day Forecast")
plt.ylabel("MAE")

In [ ]:
## Load XGBoost models if already exist
# y_pred_models = joblib.load("xgboost_50Splits_maxdepth5_models.pkl")

In [ ]:
def get_booster_importances(models, importance_type="weight"):
    """
    models: dict of {day: Booster}
    importance_type: 'gain', 'weight', or 'cover'
    """
    all_importances = {}

    for day, booster in models.items():
        # Returns only features used in at least one split
        raw_imp = booster.get_score(importance_type=importance_type)

        # Reindex to full feature list, filling missing with 0
        imp = pd.Series(raw_imp)
        all_importances[f"day_{day}"] = imp

    return pd.DataFrame(all_importances)


df_imp = get_booster_importances(y_pred_models[-1])

In [ ]:
df_imp

In [ ]:
def get_top_n_importances(models, importance_type, top_n=50):
    all_importances = {}
    for day, model in models.items():
        imp = pd.Series(model.get_score(importance_type=importance_type))
        all_importances[f"day_{day}"] = imp

    df = pd.DataFrame(all_importances)

    top_features = set()
    for col in df.columns:
        top_features.update(df[col].nlargest(top_n).index)

    return df.loc[list(top_features)]


df_imp = get_top_n_importances(y_pred_models[-1], "weight", top_n=50)

In [ ]:
df_imp